In [3]:
# ===========================================================
# EXPLAINABLE AI (XAI) PIPELINE FOR ALZHEIMER’S GAMIFIED TASKS
# Based on your Assessments.zip CSVs (ADAS, MemTrax, Cogstate, Psychometric, EMBIC, etc.)
# ===========================================================

import os
import zipfile
import pandas as pd
import numpy as np
import shap
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report

# -----------------------------
# 1. Extract ZIP and Load CSVs
# -----------------------------
zip_path = "Assessments.zip"
extract_dir = "data_extracted"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# Collect all CSVs
csv_files = [f for f in os.listdir(extract_dir) if f.endswith(".csv")]
print("CSV files found:", csv_files)

# Example: load and concatenate relevant files
dfs = []
for file in csv_files:
    path = os.path.join(extract_dir, file)
    df = pd.read_csv(path)
    df["Source"] = file  # Keep track of origin
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
print("Combined Shape:", data.shape)
print("Columns:", data.columns.tolist())

# -----------------------------
# 2. Feature Selection
# -----------------------------
# Keep only meaningful cognitive features
# Example columns expected: Reaction_Time, Accuracy, Word_Recall, Age, etc.
# Adjust based on your actual CSVs
selected_features = [
    "Age", 
    "Reaction_Time", 
    "Accuracy", 
    "Word_Recall_Score", 
    "Learning_Score"
]

# Filter dataset
df_filtered = data[[col for col in selected_features if col in data.columns] + ["DIAGNOSIS"]].dropna()

print("Filtered Columns:", df_filtered.columns.tolist())

# Encode target
le = LabelEncoder()
df_filtered["DIAGNOSIS"] = le.fit_transform(df_filtered["DIAGNOSIS"])  
# Healthy=0, MCI=1, AD=2 (example)

X = df_filtered.drop("DIAGNOSIS", axis=1)
y = df_filtered["DIAGNOSIS"]

# -----------------------------
# 3. Preprocessing
# -----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -----------------------------
# 4. Train ML Model
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_train, y_train)

print("\nClassification Report:\n", classification_report(y_test, model.predict(X_test), target_names=le.classes_))

# -----------------------------
# 5. SHAP EXPLAINABILITY
# -----------------------------
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Global Feature Importance
print("\n=== SHAP Global Feature Importance ===")
shap.summary_plot(shap_values, X_test, feature_names=X.columns)

# Local Explanation (Patient-level)
sample_idx = 5
shap.force_plot(
    explainer.expected_value[1],
    shap_values[1][sample_idx,:], 
    X_test[sample_idx,:],
    feature_names=X.columns,
    matplotlib=True
)

# -----------------------------
# 6. LIME EXPLAINABILITY
# -----------------------------
explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train,
    training_labels=y_train,
    feature_names=X.columns,
    class_names=le.classes_,
    mode="classification"
)

# Explain one prediction
exp = explainer_lime.explain_instance(X_test[5], model.predict_proba, num_features=5)
exp.show_in_notebook(show_all=False)

# -----------------------------
# 7. Example Interpretations
# -----------------------------
"""
Example Output:
- Reaction_Time ↑ (slower) → pushes prediction towards AD.
- Accuracy ↓ (fewer correct answers) → increases AD risk.
- Word_Recall_Score ↓ → strongly linked to MCI/AD.
- Younger Age → protective (pushes prediction towards Healthy).
"""


CSV files found: ['ADAS_08Jul2025.csv', 'ADNI_CBBRESULTS_08Jul2025.csv', 'ADNI_EMBICDCB_08Jul2025.csv', 'ADSXLIST_08Jul2025.csv', 'AMNART_08Jul2025.csv', 'BHR_08Jul2025.csv', 'BHR_BASELINE_QUESTIONNAIRE_08Jul2025.csv', 'BHR_EVERYDAY_COGNITION_08Jul2025.csv', 'BHR_LONGITUDINAL_QUESTIONNAIRE_08Jul2025.csv', 'BHR_MEMTRAX_08Jul2025.csv', 'BHR_SP_ADL_08Jul2025.csv', 'BHR_SP_CAREGIVER_BURDEN_08Jul2025.csv', 'BHR_SP_EVERYDAY_COGNITION_08Jul2025.csv', 'BHR_SP_FAQ_08Jul2025.csv', 'BHR_SP_INITIAL_08Jul2025.csv', 'BHR_SP_RELATIONSHIP_08Jul2025.csv', 'BHR_SP_STUDY_CONFIRMATION_08Jul2025.csv', 'BLCHANGE_08Jul2025.csv', 'CBBCOMP_08Jul2025.csv', 'CCI_08Jul2025.csv', 'CDR_08Jul2025.csv', 'CSSRSAD_08Jul2025.csv', 'DXSUM_08Jul2025.csv', 'ECOG12PT_08Jul2025.csv', 'ECOG12SP_08Jul2025.csv', 'ECOGPT_08Jul2025.csv', 'ECOGSP_08Jul2025.csv', 'EMBICqCP_08Jul2025.csv', 'FAQ_08Jul2025.csv', 'FCI_08Jul2025.csv', 'GDSCALE_08Jul2025.csv', 'IES_08Jul2025.csv', 'ITEM.csv', 'MMSE_08Jul2025.csv', 'MOCA_08Jul2025.csv', '

C:\Users\acer\AppData\Local\Temp\ipykernel_7416\1258493159.py:36: DtypeWarning: Columns (79,80) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
C:\Users\acer\AppData\Local\Temp\ipykernel_7416\1258493159.py:36: DtypeWarning: Columns (164,165) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Combined Shape: (242268, 1943)
Columns: ['PHASE', 'PTID', 'RID', 'VISCODE', 'VISCODE2', 'VISDATE', 'TOTSCORE', 'TOTAL13', 'ID', 'SITEID', 'USERDATE', 'USERDATE2', 'DD_CRF_VERSION_LABEL', 'LANGUAGE_CODE', 'HAS_QC_ERROR', 'update_stamp', 'Source', 'ProtocolID', 'SessionID', 'EXAMDATE', 'TestTime', 'Visit', 'SessionAttempt', 'SessionDuration', 'SessionCompletionPass', 'SessionPerformancePass', 'SessionIntegrityPass', 'TestCode', 'TestAttempt', 'TestCompletionScore', 'TestPerformanceScore', 'TestCompletionPass', 'TestPerformancePass', 'TestIntegrityPass', 'TestDuration', 'ReactionTime', 'RawReactionTime', 'RTVariability', 'RawRTVariability', 'Accuracy', 'RawAccuracy', 'TotalCorrect', 'TotalCorrectExclPant', 'TotalErrors', 'LegalErrors', 'RuleBreakErrors', 'TotalAnticipatory', 'TotalPost', 'TotalMaxOut', 'TotalResponses', 'TotalTrials', 'StandardScoreZ', 'StandardScoreT', 'AltStandardScoreZ', 'AltStandardScoreT', 'Psyattstdscr', 'LearnWMStdScr', 'AltLearnWMStdScr', 'N1', 'N2', 'N3', 'N4', '

ValueError: Found array with 0 sample(s) (shape=(0, 1)) while a minimum of 1 is required by StandardScaler.

In [6]:
# ===========================================================
# EXPLAINABLE AI (XAI) PIPELINE FOR ALZHEIMER’S GAMIFIED TASKS
# Based on Assessments.zip CSVs (ADAS, MemTrax, Cogstate, Psychometric, EMBIC, etc.)
# ===========================================================

import os
import zipfile
import pandas as pd
import numpy as np
import shap
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1. Extract ZIP and Load CSVs
# -----------------------------
zip_path = "Assessments.zip"
extract_dir = "data_extracted"

if not os.path.exists(extract_dir):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

# Collect all CSVs
csv_files = [f for f in os.listdir(extract_dir) if f.endswith(".csv")]
print("CSV files found:", csv_files)

dfs = []
for file in csv_files:
    path = os.path.join(extract_dir, file)
    try:
        df = pd.read_csv(path)
        df["Source"] = file  # Track origin
        dfs.append(df)
    except Exception as e:
        print(f"Skipping {file} due to error: {e}")

data = pd.concat(dfs, ignore_index=True)
print("Combined Shape:", data.shape)

# -----------------------------
# 2. Feature Selection
# -----------------------------
selected_features = [
    "Age", 
    "Reaction_Time", 
    "Accuracy", 
    "Word_Recall_Score", 
    "Learning_Score"
]

# Keep only columns that exist in dataset + DIAGNOSIS
available_features = [col for col in selected_features if col in data.columns]
if "DIAGNOSIS" not in data.columns:
    raise ValueError("DIAGNOSIS column not found in dataset. Please verify.")

df_filtered = data[available_features + ["DIAGNOSIS"]].dropna()
print("Filtered Columns:", df_filtered.columns.tolist())

# Encode target labels
le = LabelEncoder()
df_filtered["DIAGNOSIS"] = le.fit_transform(df_filtered["DIAGNOSIS"])

X = df_filtered.drop("DIAGNOSIS", axis=1)
y = df_filtered["DIAGNOSIS"]

# -----------------------------
# 3. Preprocessing
# -----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -----------------------------
# 4. Train ML Model
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_train, y_train)

print("\nClassification Report:\n", classification_report(
    y_test, model.predict(X_test), target_names=le.classes_)
)

# -----------------------------
# 5. SHAP EXPLAINABILITY
# -----------------------------
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Global feature importance
print("\n=== SHAP Global Feature Importance ===")
shap.summary_plot(shap_values, X_test, feature_names=X.columns)

# Local (patient-level) explanation
sample_idx = 5
shap.force_plot(
    explainer.expected_value[1],
    shap_values[1][sample_idx,:],
    X_test[sample_idx,:],
    feature_names=X.columns,
    matplotlib=True
)

# -----------------------------
# 6. LIME EXPLAINABILITY
# -----------------------------
explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train),
    training_labels=np.array(y_train),
    feature_names=X.columns.tolist(),
    class_names=le.classes_.tolist(),
    mode="classification"
)

exp = explainer_lime.explain_instance(
    data_row=X_test[5],
    predict_fn=model.predict_proba,
    num_features=5
)
exp.show_in_notebook(show_all=False)

# -----------------------------
# 7. Example Interpretations
# -----------------------------
"""
Example Interpretations:
- Reaction_Time ↑ (slower) → pushes prediction towards AD.
- Accuracy ↓ (fewer correct answers) → increases AD risk.
- Word_Recall_Score ↓ → strongly linked to MCI/AD.
- Younger Age → protective (pushes prediction towards Healthy).
"""


CSV files found: ['ADAS_08Jul2025.csv', 'ADNI_CBBRESULTS_08Jul2025.csv', 'ADNI_EMBICDCB_08Jul2025.csv', 'ADSXLIST_08Jul2025.csv', 'AMNART_08Jul2025.csv', 'BHR_08Jul2025.csv', 'BHR_BASELINE_QUESTIONNAIRE_08Jul2025.csv', 'BHR_EVERYDAY_COGNITION_08Jul2025.csv', 'BHR_LONGITUDINAL_QUESTIONNAIRE_08Jul2025.csv', 'BHR_MEMTRAX_08Jul2025.csv', 'BHR_SP_ADL_08Jul2025.csv', 'BHR_SP_CAREGIVER_BURDEN_08Jul2025.csv', 'BHR_SP_EVERYDAY_COGNITION_08Jul2025.csv', 'BHR_SP_FAQ_08Jul2025.csv', 'BHR_SP_INITIAL_08Jul2025.csv', 'BHR_SP_RELATIONSHIP_08Jul2025.csv', 'BHR_SP_STUDY_CONFIRMATION_08Jul2025.csv', 'BLCHANGE_08Jul2025.csv', 'CBBCOMP_08Jul2025.csv', 'CCI_08Jul2025.csv', 'CDR_08Jul2025.csv', 'CSSRSAD_08Jul2025.csv', 'DXSUM_08Jul2025.csv', 'ECOG12PT_08Jul2025.csv', 'ECOG12SP_08Jul2025.csv', 'ECOGPT_08Jul2025.csv', 'ECOGSP_08Jul2025.csv', 'EMBICqCP_08Jul2025.csv', 'FAQ_08Jul2025.csv', 'FCI_08Jul2025.csv', 'GDSCALE_08Jul2025.csv', 'IES_08Jul2025.csv', 'ITEM.csv', 'MMSE_08Jul2025.csv', 'MOCA_08Jul2025.csv', '

ValueError: Found array with 0 sample(s) (shape=(0, 1)) while a minimum of 1 is required by StandardScaler.

In [9]:
import pandas as pd
import glob
import os

# Path to extracted CSVs
path = "data_extracted"

# Load all CSV files
all_files = glob.glob(os.path.join(path, "*.csv"))

# Merge them (aligning by patient ID if available)
dfs = []
for file in all_files:
    try:
        df_temp = pd.read_csv(file)
        dfs.append(df_temp)
    except Exception as e:
        print(f"Skipping {file}, error: {e}")

# Concatenate (you may want to merge on PTID or RID instead if available)
combined_df = pd.concat(dfs, axis=0, ignore_index=True)

# Save for later use
combined_df.to_csv("combined_dataset.csv", index=False)
print("Combined dataset saved with shape:", combined_df.shape)


Combined dataset saved with shape: (242268, 1942)


In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import shap

# -----------------------------
# 1. Load combined dataset
# -----------------------------
# df = pd.read_csv("combined_dataset.csv")   # Replace with your merged file
df = pd.read_csv("combined_dataset.csv", usecols=["Accuracy", "DIAGNOSIS"])

# -----------------------------
# 2. Select useful columns
# -----------------------------
# Example: we take cognitive test accuracy + diagnosis label
if "Accuracy" in df.columns and "DIAGNOSIS" in df.columns:
    data = df[["Accuracy", "DIAGNOSIS"]].dropna()
else:
    raise ValueError("Check column names. Available: ", df.columns.tolist())

X = data[["Accuracy"]]
y = data["DIAGNOSIS"]

# Encode target
le = LabelEncoder()
y = le.fit_transform(y)

# -----------------------------
# 3. Preprocessing
# -----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -----------------------------
# 4. Train ML Model
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Model Performance:")
print(classification_report(y_test, model.predict(X_test), target_names=le.classes_))

# -----------------------------
# 5. Explainable AI (SHAP)
# -----------------------------
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Summary plot
shap.summary_plot(shap_values, X_test, feature_names=["Accuracy"])

# Force plot (first prediction explanation)
shap.initjs()
shap.force_plot(explainer.expected_value[0], shap_values[0][0], feature_names=["Accuracy"])


ValueError: Found array with 0 sample(s) (shape=(0, 1)) while a minimum of 1 is required by StandardScaler.